# Earth environmental EDA — five-site climate comparison

Ported from `notebooks/exploration/27_five_site_environment_eda.ipynb`. Original source files are unchanged. Inputs and all generated data stay under `darrel stuff/data/`.

Run local notebooks 03 and 26 first. Copies the original native coverage, seasonality, spatial variation and annual precipitation analysis. OSM/support-zone sections are excluded. This also covers the Earth climate questions from notebook 25.

**Code copied first; not executed locally.** No historical outputs are presented as new results. Install the original processing dependencies and read `../README.md` before Run All.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or one of its subfolders")
DATA_ROOT = ROOT / "darrel stuff/data"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "darrel stuff/EDA"))
from environment_workflow import LocalRun as Run, latest_completed_run, select_climate_manifest

print("Source data:", DATA_ROOT)
print("Processed outputs:", DATA_ROOT / "processed")


## Accepted local inputs and provenance

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import MaxNLocator
from notebooks.shared.inspection import check_input, file_hash, verify_run

# None selects the latest completed LOCAL run; set explicit IDs to pin a rerun.
CLIMATE_RUN_ID = None
EXPANSION_RUN_ID = None
latest_completed_run(ROOT, "earth_climate", CLIMATE_RUN_ID)
latest_completed_run(ROOT, "earth_environment_expansion", EXPANSION_RUN_ID)
ANALYSIS = Run(ROOT, "analysis/earth_environment_eda", {
    "source": "notebooks/exploration/27_five_site_environment_eda.ipynb",
    "scope": "Original climate analysis only; OSM excluded",
    "climate_run_id": CLIMATE_RUN_ID, "expansion_run_id": EXPANSION_RUN_ID,
}, "Earth/27_five_site_environment_eda.ipynb")
ANALYSIS.__enter__()
OUT = ANALYSIS.directory
expected = {}
evidence = {}

def pin(path):
    evidence[str(path.relative_to(ROOT))] = file_hash(path)
    ANALYSIS.input(path)
    return path

def register(stage, run_id=None):
    folder = latest_completed_run(ROOT, stage, run_id)
    expected.update(verify_run(folder))
    pin(folder / "provenance.json")
    return folder

def checked(path):
    audit = {}
    check_input(path, expected, audit)
    evidence[str(path.relative_to(ROOT))] = audit[str(path.resolve())]
    ANALYSIS.input(path)
    return path

def savefig(fig, name):
    fig.savefig(OUT / (name + ".png"), dpi=150, bbox_inches="tight")
    fig.savefig(OUT / (name + ".pdf"), bbox_inches="tight")
    display(fig)
    plt.close(fig)

original_climate = register("earth_climate", CLIMATE_RUN_ID)
expansion = register("earth_environment_expansion", EXPANSION_RUN_ID)
pin(ROOT / "notebooks/exploration/27_five_site_environment_eda.ipynb")
AREAS = {'atacama_san_pedro': (-68.6, -23.2, -67.8, -22.6), 'death_valley': (-117.10, 36, -116.60, 36.5),
         'lanzarote': (-13.95, 28.8, -13.35, 29.3), 'northern_arizona': (-112.5, 35, -111, 36),
         'wadi_rum': (35.2, 29.4, 35.65, 29.8)}
ADDITIONS = {'death_valley', 'wadi_rum'}
variables = ['tmin', 'tmax', 'ppt', 'ws', 'srad']
units = {'tmin': 'degC', 'tmax': 'degC', 'ppt': 'mm', 'ws': 'm/s', 'srad': 'W/m^2'}
dates = pd.date_range('2015-01-01', '2024-12-01', freq='MS')
climate = pd.concat([pd.read_csv(checked(folder / 'regional_monthly_summary.csv'), parse_dates=['date'])
                     for folder in [original_climate, expansion]], ignore_index=True)
expected_keys = pd.MultiIndex.from_product([list(AREAS), dates, variables], names=['area', 'date', 'variable'])
actual_keys = pd.MultiIndex.from_frame(pd.DataFrame(climate[['area', 'date', 'variable']]))
assert len(climate) == 3000 and actual_keys.is_unique and expected_keys.difference(actual_keys).empty
assert (climate.units == climate.variable.map(units)).all()
assert np.isfinite(climate.spatial_weighted_mean).all()
climate['month'], climate['year'] = climate.date.dt.month, climate.date.dt.year
climate.to_csv(OUT / 'five_site_monthly_summary.csv', index=False)


## Native climate completeness, seasonal and spatial variation

In [ ]:
coverage_rows = []
spatial_rows = []
cell_maps = {}
for area in AREAS:
    folder = expansion if area in ADDITIONS else original_climate
    paths = sorted((folder / 'earth' / area / 'native_monthly').glob('*.csv.gz'))
    assert len(paths) == 50
    observations = pd.concat([pd.read_csv(checked(path), parse_dates=['date']) for path in paths], ignore_index=True)
    assert observations.valid.eq(np.isfinite(observations.value)).all()
    assert not observations.duplicated(['longitude', 'latitude', 'date', 'variable']).any()
    assert set(observations.date) == set(dates) and set(observations.variable) == set(variables)
    assert observations.units.eq(observations.variable.map(units)).all()
    matrix = observations.pivot(index=['longitude', 'latitude'], columns=['date', 'variable'], values='value')
    matrix = matrix.reindex(columns=pd.MultiIndex.from_product([dates, variables], names=['date', 'variable']))
    finite = np.isfinite(matrix.to_numpy()).reshape(len(matrix), 120, 5)
    joint = np.asarray(finite.all(axis=2))
    complete = joint.all(axis=1)
    weight = np.cos(np.deg2rad(matrix.index.get_level_values('latitude').to_numpy()))
    for variable in variables:
        subset = observations.loc[observations.variable == variable]
        assert len(subset) == len(matrix) * 120
        if variable in ['ppt', 'ws', 'srad']:
            assert subset.value.dropna().ge(0).all()
    coverage_rows.append({'area': area, 'selected_native_centers': len(matrix),
                         'joint_complete_120_month_centers': int(complete.sum()),
                         'minimum_joint_month_centers': int(joint.sum(axis=0).min()),
                         'maximum_joint_month_centers': int(joint.sum(axis=0).max()),
                         'weighted_complete_fraction': float(weight[complete].sum() / weight.sum())})
    cells = matrix.index.to_frame(index=False)
    cells['joint_months_of_120'] = joint.sum(axis=1)
    # Retain missing coastal centers; no fill or interpolation.
    mean_tmax = observations.loc[observations.variable == 'tmax'].groupby(['longitude', 'latitude']).value.mean()
    cells['mean_of_120_monthly_tmax_degC'] = mean_tmax.reindex(matrix.index).to_numpy()
    cells.to_csv(OUT / f'native_climate_cells_{area}.csv', index=False)
    cell_maps[area] = cells
    for _, group in observations.groupby(['date', 'variable']):
        valid = group.loc[group.valid]
        date, variable = group.date.iloc[0], group.variable.iloc[0]
        mean = np.average(valid.value.to_numpy(), weights=np.cos(np.deg2rad(valid.latitude.to_numpy())))
        row = climate.loc[(climate.area == area) & (climate.date == date) & (climate.variable == variable)].iloc[0]
        assert np.isclose(mean, row.spatial_weighted_mean, rtol=1e-12, atol=1e-10)
        assert len(valid) == row.valid_cells and len(group) == row.selected_cells
        q10, median, q90 = np.quantile(valid.value, [.1, .5, .9])
        spatial_rows.append({'area': area, 'date': date, 'variable': variable, 'units': units[variable],
                             'native_q10': q10, 'native_median': median, 'native_q90': q90, 'native_q90_minus_q10': q90-q10})
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUT / 'climate_joint_coverage.csv', index=False)
spatial = pd.DataFrame(spatial_rows)
spatial.to_csv(OUT / 'climate_spatial_variation.csv', index=False)
seasonal = climate.groupby(['area', 'variable', 'units', 'month']).spatial_weighted_mean.agg(['mean', 'min', 'max']).reset_index()
seasonal.to_csv(OUT / 'monthly_climatology.csv', index=False)
contrast = seasonal.groupby(['area', 'variable', 'units'])['mean'].agg(['min', 'max']).reset_index()
contrast['seasonal_climatology_span'] = contrast['max'] - contrast['min']
spatial_mean = spatial.groupby(['area', 'variable']).native_q90_minus_q10.mean().rename('mean_within_month_spatial_10_90_span').reset_index()
contrast = contrast.merge(spatial_mean, on=['area', 'variable'], validate='one_to_one')
contrast.to_csv(OUT / 'seasonal_and_spatial_spans.csv', index=False)
annual_precip = climate.loc[climate.variable == 'ppt'].groupby(['area', 'year']).spatial_weighted_mean.sum().rename('annual_precipitation_mm').reset_index()
assert len(annual_precip) == 50
annual_precip.to_csv(OUT / 'annual_precipitation.csv', index=False)
fig, axes = plt.subplots(5, 5, figsize=(18, 12), sharex=True, sharey='row', layout='constrained')
for col, area in enumerate(AREAS):
    for row_index, variable in enumerate(variables):
        subset = seasonal.loc[(seasonal.area == area) & (seasonal.variable == variable)]
        ax = axes[row_index, col]
        ax.fill_between(subset.month, subset['min'], subset['max'], color='#c3dcd7')
        ax.plot(subset.month, subset['mean'], color='#176b72')
        ax.set_xticks([1, 4, 7, 10, 12])
        ax.grid(alpha=.2)
        if row_index == 0:
            ax.set_title(area.replace('_', ' '), fontsize=11)
        if col == 0:
            ax.set_ylabel(f'{variable} ({units[variable]})')
        if row_index == 4:
            ax.set_xlabel('Calendar month')
fig.suptitle('Five-site Earth climate, 2015–2024\nLine: monthly climatology of regional means; shading: between-year range, not weather extremes', fontsize=15)
savefig(fig, 'five_site_seasonality')
fig, axes = plt.subplots(1, 5, figsize=(17, 4.5), layout='constrained')
all_tmax = np.concatenate([frame.mean_of_120_monthly_tmax_degC.to_numpy() for frame in cell_maps.values()])
mesh = None
for ax, (area, cells) in zip(axes, cell_maps.items(), strict=True):
    grid = cells.pivot(index='latitude', columns='longitude', values='mean_of_120_monthly_tmax_degC').sort_index()
    mesh = ax.pcolormesh(grid.columns.to_numpy(), grid.index.to_numpy(), grid.to_numpy(), shading='nearest', cmap='magma',
                        vmin=np.nanmin(all_tmax), vmax=np.nanmax(all_tmax))
    west, south, east, north = AREAS[area]
    ax.set(xlim=(west, east), ylim=(south, north), title=area.replace('_', ' '), xlabel='Longitude')
    ax.set_aspect(1 / np.cos(np.deg2rad((south+north)/2)))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.tick_params(labelsize=8)
axes[0].set_ylabel('Latitude')
assert mesh is not None
fig.colorbar(mesh, ax=axes, label='Mean of 120 monthly average maximum temperatures (degC)', shrink=.8)
fig.suptitle('Native climate grid: spatial variation and persistent gaps\nWhite: no selected valid cell footprint. No terrain-grid interpolation.')
savefig(fig, 'native_climate_spatial_context')
display(coverage)

## Interpretation

Monthly climatology and between-year ranges are not daily extremes. Native-center coverage is not exact land-area coverage. Keep hemispheric seasonal context visible. No operating window, access or equipment feasibility is established.

Run the final cell only after all analysis cells succeed. Interrupted runs remain incomplete.

In [ ]:
assert len(climate) == 3000 and len(coverage) == 5 and len(annual_precip) == 50
required = ["five_site_monthly_summary.csv", "climate_joint_coverage.csv", "climate_spatial_variation.csv",
            "monthly_climatology.csv", "seasonal_and_spatial_spans.csv", "annual_precipitation.csv"]
for name in required:
    if not (OUT / name).is_file():
        raise FileNotFoundError(f"Run all analysis cells before finalizing: {name}")
ANALYSIS.record["interpretation"] = {
    "temperature": "Monthly average minimum/maximum air temperatures; not weather extremes",
    "coverage": "Selected native cell centers, including missing coastal/ocean support",
    "comparison": "Regional climate context, not equipment suitability or a 30 m terrain field",
}
ANALYSIS.__exit__(None, None, None)
print("Completed local environmental analysis:", OUT)
